# tSCS EMG — the four comparison figures (subject NTA, 24-07-2026)

One session, one participant: **stimulation mode** (30 Hz burst / ARC-EX) × **polarity**
(cathodic / anodic) × **lidocaine** (before / with). Four figures, each holding one thing fixed:

| figure | fixed | compared | with |
|---|---|---|---|
| **1** | anodic | 30 Hz vs ARC-EX | before and with lidocaine |
| **2** | cathodic | 30 Hz vs ARC-EX | before and with lidocaine |
| **3** | ARC-EX | anodic vs cathodic | before and with lidocaine |
| **4** | 30 Hz burst | anodic vs cathodic | before and with lidocaine |

## Colours and styles
**gray = before lidocaine, orange = with lidocaine** in every figure. The second factor is the
**style**: in figures 1–2 **solid / plain bars = 30 Hz burst, dashed / hatched = ARC-EX**; in
figures 3–4 **solid / plain = cathodic, dashed / hatched = anodic**.

**Intensities:** `BURST_MA` and `ARCEX_MA` in the config set the mA of 30 Hz and of ARC-EX in
every figure; the `AMP[...]` lines below them override any single train (polarity × lidocaine)
when you want each at its own motor threshold — the log gives burst cathodic 25 mA before / 30
with lidocaine, anodic 30 / 30; ARC-EX cathodic 70 / 70, anodic 65 / 90. `FIGS[n]` sets the muscle
of each figure's one-muscle version (and can override its intensities too).

Every figure also comes with **waterfalls** — all intensities of each condition stacked, and the
four overlaid — so you can see the whole sweep before choosing the mA to compare at.

In [ ]:
# run from the repo root so that src/, results/ and tSCS_CHUV_data/ resolve the same way from
# every notebook folder (VS Code starts the kernel in the notebook's own folder)
import os, sys
while not os.path.isdir("src") and os.getcwd() != "/":
    os.chdir("..")
sys.path.insert(0, os.path.abspath("src"))


In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt

from functions import set_style, load_run, pretty, waterfall, waterfall_overlay, detect_pulses
from functions.burst import (compare_at_intensity, summary_curves, plot_pulse_overlay,
                             burst_p2p, detection_report, resolve_muscles, artifact_extent)
from functions.paper import fig_train_modes
set_style()


## Config

In [ ]:
D = "tSCS_CHUV_data/24-07-2026/testSCS/"
FILE = {   # (mode, polarity, lidocaine state) -> file
    ("burst", "cathodic", "before"):    "Burst_autosave_20260724_102443_670ms.csv",
    ("burst", "cathodic", "lidocaine"): "Burst_autosave_20260724_113834_522ms.csv",
    ("burst", "anodic",   "before"):    "Burst_autosave_20260724_102721_694ms.csv",
    ("burst", "anodic",   "lidocaine"): "Burst_autosave_20260724_114055_891ms.csv",
    ("arcex", "cathodic", "before"):    "Modulated_autosave_20260724_103530_508ms.csv",
    ("arcex", "cathodic", "lidocaine"): "Modulated_autosave_20260724_114332_473ms.csv",
    ("arcex", "anodic",   "before"):    "Modulated_autosave_20260724_103849_459ms.csv",
    ("arcex", "anodic",   "lidocaine"): "Modulated_autosave_20260724_114730_561ms.csv",
}
NAME = {"burst": "30 Hz", "arcex": "ARC-EX", "cathodic": "cathodic", "anodic": "anodic"}
LIDO_COL = {"before": "0.45", "lidocaine": "#f39c12"}       # colour = lidocaine
STYLE_A  = ("",    "-")                                      # first compared value: plain / solid
STYLE_B  = ("///", "--")                                     # second: hatched / dashed

# ---- the intensity of every train, in one table -------------------------------------------
BURST_MA, ARCEX_MA = 35, 110      # <-- the mA used for 30 Hz and for ARC-EX, everywhere
AMP = {k: (BURST_MA if k[0] == "burst" else ARCEX_MA) for k in FILE}

# per-condition overrides - uncomment/edit any line to give that train its own mA
# AMP[("burst", "cathodic", "before")]    = 30      # log motor threshold: 25
# AMP[("burst", "cathodic", "lidocaine")] = 30
# AMP[("burst", "anodic",   "before")]    = 30
# AMP[("burst", "anodic",   "lidocaine")] = 30
# AMP[("arcex", "cathodic", "before")]    = 70
# AMP[("arcex", "cathodic", "lidocaine")] = 70
# AMP[("arcex", "anodic",   "before")]    = 65
# AMP[("arcex", "anodic",   "lidocaine")] = 90

FIGS = {   # what each figure holds fixed and what it compares (the mA and the muscle are chosen
           # in the small settings cell right before each figure)
    1: dict(fixed=("polarity", "anodic"),   compare=("burst", "arcex")),
    2: dict(fixed=("polarity", "cathodic"), compare=("burst", "arcex")),
    3: dict(fixed=("mode", "arcex"),        compare=("cathodic", "anodic")),
    4: dict(fixed=("mode", "burst"),        compare=("cathodic", "anodic")),
}
XLIM_WF   = (-20, 130)      # time window of the waterfalls (ms)
WF_EACH   = True            # also draw each condition's waterfall on its own, not only the overlay
WF_MUSCLES = None           # None = all muscles in the waterfalls, or a list, e.g. ["Flex. digitorum (R)"]

# ---- peak-detection parameters (see the section "How the peaks are detected" below) --------
N_PULSES      = 10      # pulses of the train analysed
RESP_START_MS = 8.0     # response window starts this long after EACH pulse onset - must clear the
                        # artifact. A dict overrides single channels: {"R_DELmed": 11.0}
RESP_END_MS   = None    # window end, also from the pulse onset; None = run to the next pulse
GUARD_MS      = 1.0     # ...minus this
MIN_SNR       = 2.0     # keep a train only if pulse-1 p2p >= MIN_SNR x baseline p2p (None = keep all)
MAX_EDGE_FRAC = 0.5     # reject a train as artifact if more than this fraction of its pulses have
                        # max/min on a window border (None = off)
EDGE_MS       = 1.0     # "on a border" means within this many ms of it
JITTER_MS     = 3.0     # flag a pulse whose peak latency differs from the train median by more than this
KW = dict(n_pulses=N_PULSES, resp_start_ms=RESP_START_MS, resp_end_ms=RESP_END_MS,
          guard_ms=GUARD_MS, min_snr=MIN_SNR, max_edge_frac=MAX_EDGE_FRAC)

muscles = [c for c in load_run(D + FILE[("burst", "cathodic", "before")])[2] if c != "Trigger A"]

def build(n, amps=None, muscle=None):
    """Everything figure `n` needs, ordered A-before, A-lidocaine, B-before, B-lidocaine.
    amps: {compared value: mA} for this figure (default: the AMP table above).
    muscle: the muscle of the one-muscle version and of the focused waterfalls."""
    spec = FIGS[n]; fixed, compare = spec["fixed"], spec["compare"]
    files, labels, colours, hatches, lss, amp_list, keys = [], [], [], [], [], [], []
    for val, (hat, ls) in zip(compare, (STYLE_A, STYLE_B)):
        for state in ("before", "lidocaine"):
            key = (val, fixed[1], state) if fixed[0] == "polarity" else (fixed[1], val, state)
            keys.append(key); files.append(D + FILE[key])
            labels.append(f"{NAME[key[0]]} · {key[1]} · {'before lido' if state == 'before' else 'with lido'}")
            colours.append(LIDO_COL[state]); hatches.append(hat); lss.append(ls)
            amp_list.append((amps or {}).get(val, AMP[key]))
    return dict(files=files, labels=labels, colours=colours, hatches=hatches, linestyles=lss,
                amps=tuple(amp_list), keys=keys, muscle=muscle, n=n)

def peaks(cfg, muscle=None, report=True, kw=None):
    """Look at HOW the peaks are detected for this figure, at the mA it uses.
    One pulse-overlay per condition: every pulse re-aligned to its own onset, green = the response
    window, v / ^ = the max / min taken, red ring = flagged (on a window border, or at a different
    latency than the other pulses). Stacked markers = the same deflection every time.
    muscle: one label / list (default: the figure's muscle) · kw: override the detection parameters."""
    kw = kw or KW
    ms = muscle or cfg["muscle"] or muscles
    for key, f_, a in zip(cfg["keys"], cfg["files"], cfg["amps"]):
        meta_, t_, sig_ = load_run(f_)
        lab = f"{NAME[key[0]]} · {key[1]} · {'before lido' if key[2] == 'before' else 'with lido'}"
        plot_pulse_overlay(meta_, t_, sig_, ms, amp=a, edge_ms=EDGE_MS, jitter_ms=JITTER_MS,
                           title=f"{lab} — {a} mA", **kw)
        if report:
            chans = resolve_muscles([c for c in sig_ if c != "Trigger A"], ms if isinstance(ms, list) else [ms])
            res = burst_p2p(meta_, t_, sig_, chans, **kw)
            print(f"----- {lab} — {a} mA")
            detection_report(res, chans, edge_ms=EDGE_MS, jitter_ms=JITTER_MS)
            print(f"  artifact spike width (ms after onset): "
                  + ", ".join(f"{pretty(m)} {v:.1f}" for m, v in
                              artifact_extent(t_, sig_, chans, detect_pulses(t_, sig_["Trigger A"])[:N_PULSES],
                                              w=[x["amp_ma"] for x in meta_].index(a)).items()))

def check(cfg):
    """Print what each train of this figure will be plotted at, and whether that mA exists."""
    for key, f_, a in zip(cfg["keys"], cfg["files"], cfg["amps"]):
        avail = [m["amp_ma"] for m in load_run(f_)[0]]
        print(f"{NAME[key[0]]:7s} {key[1]:9s} {key[2]:10s} -> {a:4d} mA "
              + ("ok" if a in avail else f"MISSING - available: {avail}"))

def show(cfg, title, muscle=None):
    """The comparison figure; muscle=cfg["muscle"] for the one-muscle version."""
    compare_at_intensity(cfg["files"], None, amp=cfg["amps"], normalize="none", muscles=muscle,
                         labels=cfg["labels"], colours=cfg["colours"], hatches=cfg["hatches"],
                         linestyles=cfg["linestyles"], title=title, **KW)

def wf(cfg, prefix, muscles_=None, each=None):
    """Waterfalls: every intensity of each condition, titled with mode · polarity · lidocaine,
    then all four overlaid. One shared gain per muscle, so amplitudes are comparable."""
    ms = muscles_ or WF_MUSCLES or muscles
    runs = [load_run(f) for f in cfg["files"]]
    gains = waterfall(*runs[0], ms, xlim=XLIM_WF,
                      title=f"{prefix} · waterfall — {cfg['labels'][0]}  (sets the gain)")
    if WF_EACH if each is None else each:
        for lab, r in zip(cfg["labels"][1:], runs[1:]):
            waterfall(*r, ms, xlim=XLIM_WF, gains=gains, title=f"{prefix} · waterfall — {lab}")
    waterfall_overlay(runs, muscles=ms, xlim=XLIM_WF, gains=gains, labels=cfg["labels"],
                      colours=cfg["colours"], linestyles=cfg["linestyles"],
                      title=f"{prefix} · waterfalls overlaid — all four conditions")

for k, f_ in FILE.items():                        # what each recording contains
    print(f"{NAME[k[0]]:7s} {k[1]:9s} {k[2]:10s} {[m['amp_ma'] for m in load_run(D + f_)[0]]} mA")


## How the peaks are detected — and how to check it

For every pulse the peak-to-peak is `max − min` inside a window that starts **after that pulse's
artifact**:

    window_k = [ pulse_k onset + RESP_START_MS , pulse_(k+1) onset − GUARD_MS ]   (or + RESP_END_MS)

Then two filters: a train counts as a response only if **pulse-1 p2p ≥ `MIN_SNR` × the
pre-stimulus baseline p2p**, and a train is **rejected as artifact** if more than `MAX_EDGE_FRAC`
of its pulses have their max/min sitting within `EDGE_MS` of a window border (a smooth
artifact-recovery curve has no peak inside the window, so its extremes fall on the edges).

| parameter | raise it when | lower it when |
|---|---|---|
| `RESP_START_MS` | the ▼/▲ land on the artifact decay right after the pulse | the window cuts off the start of a real response |
| `RESP_END_MS` | — | the window catches something late that isn't the response |
| `MIN_SNR` | noise-only trains are being kept | real small responses are dropped |
| `MAX_EDGE_FRAC` | a real response is being called artifact | artifact recovery is being kept |
| `JITTER_MS` | too many pulses flagged on a variable-latency muscle | you want stricter flagging |

`peaks(FIGn)` below draws, for each condition **at the mA that figure uses**, every pulse
re-aligned to its own onset: **green = the response window, ▼/▲ = the max/min actually taken, red
ring = flagged**. If the detection is right, the markers stack on top of each other. It also
prints the flag counts and the measured artifact width per channel, so you can set
`RESP_START_MS` above it.

To try other settings without touching the config, pass `kw=`:
```python
peaks(FIG1, kw=dict(KW, resp_start_ms=11.0, min_snr=3.0))
```

## Figure 1 · 30 Hz vs ARC-EX — **anodic**, before and with lidocaine

Plain bars / solid lines = **30 Hz burst**, hatched / dashed = **ARC-EX**; gray = **before lidocaine**, orange = **with lidocaine**.

**Settings for this figure** are in the next cell: the mA of each compared condition, the muscle
for the one-muscle version, and which muscles go in the waterfalls.

In [ ]:
# ======== Figure 1 settings ========
F1_AMPS   = {"burst": BURST_MA, "arcex": ARCEX_MA}      # mA of each compared condition
F1_MUSCLE = "Flex. digitorum (R)"                # muscle of the one-muscle version
F1_WF     = None                                 # waterfall muscles: None = all, or e.g. [F1_MUSCLE]
# =====================================

FIG1 = build(1, amps=F1_AMPS, muscle=F1_MUSCLE)
check(FIG1)


### Figure 1a · all muscles

In [ ]:
show(FIG1, "Fig 1 · anodic — all muscles")


### Figure 1b · one muscle

In [ ]:
show(FIG1, f"Fig 1 · anodic — {FIG1['muscle']}", muscle=FIG1["muscle"])


### Figure 1c · how the peaks are detected (at the mA above)

In [ ]:
peaks(FIG1)                      # the figure's muscle; peaks(FIG1, muscle="Biceps (R)") for another
# peaks(FIG1, kw=dict(KW, resp_start_ms=11.0))    # try other detection settings


### Figure 1d · waterfalls — every intensity of each condition

One waterfall per condition, titled **mode · polarity · before / with lidocaine**, then the four
overlaid. All share one gain per muscle, so heights are comparable.

In [ ]:
wf(FIG1, "Fig 1 · anodic", muscles_=F1_WF)


## Figure 2 · 30 Hz vs ARC-EX — **cathodic**, before and with lidocaine

Plain / solid = **30 Hz burst**, hatched / dashed = **ARC-EX**; gray = **before lidocaine**, orange = **with lidocaine**.

**Settings for this figure** are in the next cell: the mA of each compared condition, the muscle
for the one-muscle version, and which muscles go in the waterfalls.

In [ ]:
# ======== Figure 2 settings ========
F2_AMPS   = {"burst": BURST_MA, "arcex": ARCEX_MA}      # mA of each compared condition
F2_MUSCLE = "Flex. digitorum (R)"                # muscle of the one-muscle version
F2_WF     = None                                 # waterfall muscles: None = all, or e.g. [F2_MUSCLE]
# =====================================

FIG2 = build(2, amps=F2_AMPS, muscle=F2_MUSCLE)
check(FIG2)


### Figure 2a · all muscles

In [ ]:
show(FIG2, "Fig 2 · cathodic — all muscles")


### Figure 2b · one muscle

In [ ]:
show(FIG2, f"Fig 2 · cathodic — {FIG2['muscle']}", muscle=FIG2["muscle"])


### Figure 2c · how the peaks are detected (at the mA above)

In [ ]:
peaks(FIG2)                      # the figure's muscle; peaks(FIG2, muscle="Biceps (R)") for another
# peaks(FIG2, kw=dict(KW, resp_start_ms=11.0))    # try other detection settings


### Figure 2d · waterfalls — every intensity of each condition

One waterfall per condition, titled **mode · polarity · before / with lidocaine**, then the four
overlaid. All share one gain per muscle, so heights are comparable.

In [ ]:
wf(FIG2, "Fig 2 · cathodic", muscles_=F2_WF)


## Figure 3 · ARC-EX — **cathodic vs anodic**, before and with lidocaine

Plain / solid = **cathodic**, hatched / dashed = **anodic**; gray = **before lidocaine**, orange = **with lidocaine**.

**Settings for this figure** are in the next cell: the mA of each compared condition, the muscle
for the one-muscle version, and which muscles go in the waterfalls.

In [ ]:
# ======== Figure 3 settings ========
F3_AMPS   = {"cathodic": ARCEX_MA, "anodic": ARCEX_MA}      # mA of each compared condition
F3_MUSCLE = "Flex. digitorum (R)"                # muscle of the one-muscle version
F3_WF     = None                                 # waterfall muscles: None = all, or e.g. [F3_MUSCLE]
# =====================================

FIG3 = build(3, amps=F3_AMPS, muscle=F3_MUSCLE)
check(FIG3)


### Figure 3a · all muscles

In [ ]:
show(FIG3, "Fig 3 · ARC-EX — all muscles")


### Figure 3b · one muscle

In [ ]:
show(FIG3, f"Fig 3 · ARC-EX — {FIG3['muscle']}", muscle=FIG3["muscle"])


### Figure 3c · how the peaks are detected (at the mA above)

In [ ]:
peaks(FIG3)                      # the figure's muscle; peaks(FIG3, muscle="Biceps (R)") for another
# peaks(FIG3, kw=dict(KW, resp_start_ms=11.0))    # try other detection settings


### Figure 3d · waterfalls — every intensity of each condition

One waterfall per condition, titled **mode · polarity · before / with lidocaine**, then the four
overlaid. All share one gain per muscle, so heights are comparable.

In [ ]:
wf(FIG3, "Fig 3 · ARC-EX", muscles_=F3_WF)


### Figure 3e · across intensities

In [ ]:
summary_curves(FIG3["files"], None, labels=FIG3["labels"], colours=FIG3["colours"],
               markers=["o", "o", "s", "s"], **KW);


## Figure 4 · 30 Hz burst — **cathodic vs anodic**, before and with lidocaine

Plain / solid = **cathodic**, hatched / dashed = **anodic**; gray = **before lidocaine**, orange = **with lidocaine**.

**Settings for this figure** are in the next cell: the mA of each compared condition, the muscle
for the one-muscle version, and which muscles go in the waterfalls.

In [ ]:
# ======== Figure 4 settings ========
F4_AMPS   = {"cathodic": BURST_MA, "anodic": BURST_MA}      # mA of each compared condition
F4_MUSCLE = "Flex. digitorum (R)"                # muscle of the one-muscle version
F4_WF     = None                                 # waterfall muscles: None = all, or e.g. [F4_MUSCLE]
# =====================================

FIG4 = build(4, amps=F4_AMPS, muscle=F4_MUSCLE)
check(FIG4)


### Figure 4a · all muscles

In [ ]:
show(FIG4, "Fig 4 · 30 Hz burst — all muscles")


### Figure 4b · one muscle

In [ ]:
show(FIG4, f"Fig 4 · 30 Hz burst — {FIG4['muscle']}", muscle=FIG4["muscle"])


### Figure 4c · how the peaks are detected (at the mA above)

In [ ]:
peaks(FIG4)                      # the figure's muscle; peaks(FIG4, muscle="Biceps (R)") for another
# peaks(FIG4, kw=dict(KW, resp_start_ms=11.0))    # try other detection settings


### Figure 4d · waterfalls — every intensity of each condition

One waterfall per condition, titled **mode · polarity · before / with lidocaine**, then the four
overlaid. All share one gain per muscle, so heights are comparable.

In [ ]:
wf(FIG4, "Fig 4 · 30 Hz burst", muscles_=F4_WF)


### Figure 4e · across intensities

In [ ]:
summary_curves(FIG4["files"], None, labels=FIG4["labels"], colours=FIG4["colours"],
               markers=["o", "o", "s", "s"], **KW);


## Paper-style version — traces + pulse 1 vs rest, 2–3 muscles

`fig_train_modes` for any pair of conditions; set `SAVE` to write a 300 dpi PNG.

In [ ]:
MUSCLES_FIG = ["Flex. digitorum (R)", "Flex. carpi rad. (R)", "Ext. digitorum (L)"]
SAVE = None        # e.g. "figures/fig1_anodic_burst_vs_arcex.png"

specs = [dict(label="30 Hz burst", csv=D + FILE[("burst", "anodic", "before")], amp=AMP[("burst", "anodic", "before")], colour="black"),
         dict(label="ARC-EX",      csv=D + FILE[("arcex", "anodic", "before")], amp=AMP[("arcex", "anodic", "before")], colour="#d62728")]
fig_train_modes(specs, MUSCLES_FIG, n_pulses=N_PULSES, resp_start_ms=RESP_START_MS,
                title="anodic · before lidocaine", save=SAVE);
